# LifeMate — lm-A-059: Gipformer 1.5 train trên toàn bộ audio train

**Import → Save Version → Save & Run All**, T4 x2, Internet On. Input duy nhất: LifeMate dataset (version 1).

**Vì sao:** trộn Whisper + Gipformer đưa LB từ 57.12 lên 60.36, nhưng Gipformer ở bản đó mới học 2955 audio `train_fit`. Whisper đã học đủ 5036 audio.

**Làm gì:** công thức `lr15` của lm-A-031, số epoch 4 đã chọn trên dev ở run đó, train trên 5036 audio → chép lời 1933 audio public bằng greedy và beam 4. Không còn dev độc lập vì mọi audio đều đem train; đánh giá bằng LB sau khi trộn. Ước tính ~50 phút, chạy trên GPU0.

Output: `gip_full/lr15/asr_public_test_gipft_lr15.shard0.jsonl` (greedy), `..._beam.shard0.jsonl`, `model_best.pt`, `metrics.json`.


In [ ]:
CONFIG = {
    "experiment": "lm-A-059-gipformer-full",
    "seed": 2026,
    "model": "g-group-ai-lab/gipformer1.5-68M-rnnt",
    "revision": "dd9227dcd8705c13f33bdbe59728d546ab94480f",
    # sherpa-onnx greedy zero-shot trên dev (CPU, chạy trong lúc GPU train): mốc trước/sau
    "reference_models": {
        "gip10": ["g-group-ai-lab/gipformer-68M-rnnt", "318322b009200f8e77a3751fa64638e5cc85a7dc"],
        "gip15": ["g-group-ai-lab/gipformer1.5-68M-rnnt", "dd9227dcd8705c13f33bdbe59728d546ab94480f"],
    },
    "icefall_commit": "9c966762e058bd7ba79d9da71e42fc06b64b703e",
    "out_dir": "/kaggle/working/gip_full",
    "full_data": True,
    "feat_dir": "/tmp/gipft_feats",   # ~1 GB fbank, không đưa vào output
    "icefall_dir": "/tmp/icefall",
    "feat_procs": 4,
    "session_budget_s": 11.5 * 3600,
    "reserve_s": 1200,               # chừa cho holdout + public + lưu model sau epoch cuối
    "eval_zero_shot": False,
    "save_model": True,
    "variants": {                    # công thức lr15 của lm-A-031, epoch 4 đã chọn theo dev ở run đó
        "lr15": {"gpu": 0, "lr": 0.0015, "epochs": 4, "max_frames": 20000, "eval_max_frames": 60000},
    },
}


In [ ]:
import glob, importlib.util, json, os, re, socket, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path
SESSION_STARTED = time.time()
WORK = Path("/kaggle/working"); OUT = Path(CONFIG["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)

def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-2000:]); return r.returncode

def run_py(args, env=None):
    """Chạy python con, in log ra cell (subprocess.run thường in vào terminal của kernel, không hiện trong notebook)."""
    p = subprocess.Popen([sys.executable, *args], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=str(WORK))
    for line in p.stdout:
        print(line, end="", flush=True)
    return p.wait()

# Thiếu mạng thì dừng ngay (lm-A-030 lần 1 mất 41 phút retry rồi mới chết)
try:
    socket.create_connection(("huggingface.co", 443), timeout=10).close()
except OSError as e:
    raise RuntimeError("Không có Internet: Session options → Internet On, rồi Save & Run All lại") from e

import torch
TV, CU, PY = torch.__version__.split("+")[0], torch.version.cuda, f"{sys.version_info.major}{sys.version_info.minor}"
print("torch", torch.__version__, "cuda", CU, "python", PY, "gpus", torch.cuda.device_count())

# k2 (loss pruned RNN-T): wheel dựng sẵn khớp đúng torch / CUDA / Python, --no-deps để không đụng torch
html = urllib.request.urlopen("https://k2-fsa.github.io/k2/cuda.html", timeout=60).read().decode()
pat = rf'href="([^"]+/k2-[^"/]+\+cuda{re.escape(CU)}\.torch{re.escape(TV)}-cp{PY}-cp{PY}-manylinux[^"]*x86_64\.whl)"'
whl = sorted(set(re.findall(pat, html)), key=lambda u: u.rsplit("/", 1)[-1])
if not whl:
    raise RuntimeError(f"Không có wheel k2 cho torch {TV} / CUDA {CU} / cp{PY}: image Kaggle đổi, xem https://k2-fsa.github.io/k2/cuda.html")
print("k2 wheel:", whl[-1].rsplit("/", 1)[-1])
assert sh(f"{sys.executable} -m pip install -q --no-deps {whl[-1]} graphviz") == 0
# Gói nhẹ cho icefall + sherpa-onnx; ghim torch đang có để pip không thay torch
Path("/tmp/torch-pin.txt").write_text(f"torch=={torch.__version__}\n")
pkgs = "lhotse kaldialign pypinyin sherpa-onnx kaldi-native-fbank sentencepiece soundfile"
if importlib.util.find_spec("tensorboard") is None:
    pkgs += " tensorboard"
assert sh(f"{sys.executable} -m pip install -q -c /tmp/torch-pin.txt {pkgs}") == 0

# icefall (mã Zipformer, optimizer ScaledAdam, beam search) ghim commit, để ngoài /kaggle/working
ICE = Path(CONFIG["icefall_dir"])
if not (ICE / "icefall").is_dir():
    urllib.request.urlretrieve(f"https://codeload.github.com/k2-fsa/icefall/tar.gz/{CONFIG['icefall_commit']}", "/tmp/icefall.tgz")
    with tarfile.open("/tmp/icefall.tgz") as t:
        t.extractall("/tmp", filter="data")
    os.rename(f"/tmp/icefall-{CONFIG['icefall_commit']}", ICE)

from huggingface_hub import snapshot_download
MODEL_DIR = snapshot_download(CONFIG["model"], revision=CONFIG["revision"], allow_patterns=["model.pt", "bpe.model", "tokens.txt", "config.json"])
REF_DIRS = {k: snapshot_download(m, revision=r, allow_patterns=["encoder.onnx", "decoder.onnx", "joiner.onnx", "tokens.txt"])
            for k, (m, r) in CONFIG["reference_models"].items()}
print(MODEL_DIR, REF_DIRS)

# Import thử trong process con (k2 phải chạy được trên CUDA)
chk = f"""
import sys; sys.path[:0] = [{str(ICE)!r}, {str(ICE / 'egs/librispeech/ASR/zipformer')!r}]
import torch, k2, lhotse, sherpa_onnx, kaldi_native_fbank, sentencepiece
import model, zipformer, optim, beam_search
print('k2 cuda ok', k2.RaggedTensor([[1, 2], [3]]).to(torch.device('cuda')).device, 'lhotse', lhotse.__version__)
"""
assert run_py(["-c", chk]) == 0, "import icefall / k2 lỗi, xem log ở trên"
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

def find_split(split):
    hits = sorted(glob.glob(f"/kaggle/input/**/{split}/{split}.jsonl", recursive=True))
    if not hits: raise FileNotFoundError(f"không thấy {split}.jsonl: Add Input dataset thanhkhil/zaloai-lifemate-dataset")
    return Path(hits[0]).parent
TRAIN_DIR, PUBLIC_DIR = find_split("train"), find_split("public_test")
print(TRAIN_DIR, PUBLIC_DIR)

In [ ]:
%%writefile /kaggle/working/gipft.py
"""Gipformer 1.5 (Zipformer RNN-T, icefall) fine-tune worker, one variant per GPU: train on train_fit audio (history +
final turns), pick the epoch by dev WER, report final_holdout WER once, then transcribe dev / holdout / public_test with
the chosen epoch (greedy search, as the agent pipeline's sherpa-onnx backend).

    python gipft.py --config cfg.json --variant lr45
Split = the same seed-2026 manifest as lmlib.data.make_manifest (copied verbatim below, as in asrft.py).
Features = kaldi-native-fbank with sherpa-onnx's defaults, precomputed once by the notebook (`python gipft.py --feats`).
"""
from __future__ import annotations

import argparse
import hashlib
import json
import math
import random
import re
import sys
import time
import unicodedata
from collections import Counter
from pathlib import Path


# ------------------------------------------------------------------ split (verbatim from lmlib/data.py)
def audio_paths(row: dict) -> list[str]:
    out = [t["audio"] for t in row.get("history", []) if t.get("role") == "user" and t.get("audio")]
    return out + [row["audio"]]


def _template_key(row: dict) -> str:
    text = re.sub(r"\d+", "#", (row.get("audio_text") or "").lower())
    names = ",".join(sorted({c["name"] for c in (row.get("label") or {}).get("tool_calls", [])}))
    return hashlib.md5(f"{names}|{text[:40]}".encode()).hexdigest()


def label_apps(row: dict) -> set[str]:
    calls = list((row.get("label") or {}).get("tool_calls", []))
    for t in row.get("history", []):
        calls += t.get("tool_calls", []) or []
    return {c["name"].split(".", 1)[0] for c in calls if not c["name"].startswith("sys.")}


def make_manifest(rows: list[dict], seed: int = 2026, holdout_apps: int = 4, dev_frac: float = 0.12, final_frac: float = 0.12) -> dict:
    rng = random.Random(seed)
    use = Counter(a for r in rows for a in label_apps(r))
    cands = sorted(a for a, n in use.items() if 20 <= n <= 120)
    held = sorted(rng.sample(cands, min(holdout_apps, len(cands))))
    unseen = [r["id"] for r in rows if label_apps(r) & set(held)]
    rest = [r for r in rows if r["id"] not in set(unseen)]
    groups: dict[str, list[str]] = {}
    for r in rest:
        groups.setdefault(_template_key(r), []).append(r["id"])
    keys = sorted(groups)
    rng.shuffle(keys)
    n = len(rest)
    ids = {"train_fit": [], "dev_seen": [], "final_holdout": [], "dev_unseen": unseen}
    for k in keys:
        if len(ids["final_holdout"]) < final_frac * n:
            ids["final_holdout"] += groups[k]
        elif len(ids["dev_seen"]) < dev_frac * n:
            ids["dev_seen"] += groups[k]
        else:
            ids["train_fit"] += groups[k]
    return {"seed": seed, "holdout_apps": held, "ids": ids, "counts": {k: len(v) for k, v in ids.items()}}


def build_items(train_rows: list[dict], public_rows: list[dict]) -> dict:
    """{part: [{audio, text, episode, turn, part}]}; every user audio of an episode goes to the episode's part."""
    m = make_manifest(sorted(train_rows, key=lambda r: r["id"]))
    part_of = {i: p for p, ids in m["ids"].items() for i in ids}
    parts = {"train_fit": [], "dev_seen": [], "dev_unseen": [], "final_holdout": [], "public": []}
    seen = {}
    for r in train_rows:
        turns = [t for t in r.get("history", []) if t.get("role") == "user" and t.get("audio")] + [r]
        for k, t in enumerate(turns):
            turn = "final" if k == len(turns) - 1 else "history"
            if t["audio"] in seen:
                raise ValueError(f"audio shared by two turns: {t['audio']}")
            seen[t["audio"]] = 1
            parts[part_of[r["id"]]].append({"audio": t["audio"], "text": t["audio_text"], "episode": r["id"], "turn": turn,
                                            "part": part_of[r["id"]]})
    for r in public_rows:
        for p in audio_paths(r):
            parts["public"].append({"audio": p, "text": None, "episode": r["id"], "turn": "final" if p == r["audio"] else "history",
                                    "part": "public"})
    return {"manifest_counts": m["counts"], "holdout_apps": m["holdout_apps"], "parts": parts}


# ------------------------------------------------------------------ text: gipformer's vocab is upper-case words, no digits
DIGITS = "không một hai ba bốn năm sáu bảy tám chín".split()


def _under_1000(n: int, full: bool) -> list[str]:
    h, r = divmod(n, 100)
    t, u = divmod(r, 10)
    out = [DIGITS[h], "trăm"] if h or full else []
    if t == 0:
        if u:
            out += (["linh"] if out else []) + [DIGITS[u]]
    elif t == 1:
        out += ["mười"] + (["lăm" if u == 5 else DIGITS[u]] if u else [])
    else:
        out += [DIGITS[t], "mươi"] + ([{1: "mốt", 5: "lăm"}.get(u, DIGITS[u])] if u else [])
    return out


def number_words(n: int) -> list[str]:
    """123 -> một trăm hai mươi ba; 1005 -> một nghìn không trăm linh năm."""
    if n == 0:
        return ["không"]
    groups = []
    while n:
        n, g = divmod(n, 1000)
        groups.append(g)
    units = ["", "nghìn", "triệu", "tỷ"]
    if len(groups) > len(units):
        raise ValueError("number too large")
    out = []
    for i in range(len(groups) - 1, -1, -1):
        if groups[i]:
            out += _under_1000(groups[i], full=bool(out)) + ([units[i]] if units[i] else [])
    return out


def verbalize(text: str) -> str:
    """Digits -> spoken words. Phone / ticket numbers (leading 0 or >= 5 digits) and >= 3 digits glued to letters
    (flight RV107) are read digit by digit; other numbers as quantities (RON95 -> RON chín mươi lăm, 8 giờ kém 15)."""
    text = unicodedata.normalize("NFC", text or "")
    text = re.sub(r"\b0\d{2,}(?:[ .]\d{2,})+\b", lambda m: re.sub(r"[ .]", "", m.group(0)), text)  # 0983 214 6755

    def rep(m):
        s, a, b = m.group(0), m.start(), m.end()
        code = (a > 0 and text[a - 1].isalpha()) or (b < len(text) and text[b].isalpha())
        if (code and len(s) >= 3) or (len(s) > 1 and s[0] == "0") or len(s) >= 5:
            words = [DIGITS[int(c)] for c in s]
        else:
            words = number_words(int(s))
        return " " + " ".join(words) + " "

    return re.sub(r"\d+", rep, text)


def norm_text(s: str) -> str:
    t = unicodedata.normalize("NFC", s or "").lower()
    t = re.sub(r"[^\w\s]", " ", t)
    return " ".join(t.split())


def norm_spoken(s: str) -> str:
    return norm_text(verbalize(s))


def target_text(s: str) -> str:
    return norm_spoken(s).upper()


# ------------------------------------------------------------------ metrics (as asrft.py, plus the spoken-form WER)
def edit_distance(a: list, b: list) -> int:
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]


def score(items: list[dict], hyp: dict[str, str]) -> dict:
    """Corpus WER / CER / exact rate, overall and per (part, turn) slice. `wer` compares spoken forms (digits read
    out on both sides: fair to gipformer, which never writes digits); `wer_raw` is asrft.py's metric (digits kept)."""
    def agg(sel):
        we = wn = re_ = rn = ce = cn = ex = 0
        for it in sel:
            g, h = norm_spoken(it["text"]), norm_spoken(hyp.get(it["audio"], ""))
            we += edit_distance(g.split(), h.split()); wn += len(g.split())
            ce += edit_distance(list(g.replace(" ", "")), list(h.replace(" ", ""))); cn += len(g.replace(" ", ""))
            ex += g == h
            gr, hr = norm_text(it["text"]).split(), norm_text(hyp.get(it["audio"], "")).split()
            re_ += edit_distance(gr, hr); rn += len(gr)
        return {"n": len(sel), "wer": we / max(wn, 1), "wer_raw": re_ / max(rn, 1), "cer": ce / max(cn, 1),
                "exact": ex / max(len(sel), 1)}
    out = {"all": agg(items), "final": agg([i for i in items if i["turn"] == "final"])}
    for part in sorted({i["part"] for i in items}):
        out[f"{part}/final"] = agg([i for i in items if i["part"] == part and i["turn"] == "final"])
    return out


# ------------------------------------------------------------------ audio / features
def read_wave(path: Path):
    import numpy as np
    import soundfile as sf

    w, sr = sf.read(str(path), dtype="float32")
    if w.ndim > 1:
        w = w.mean(axis=1)
    if sr != 16000:
        import librosa

        w = librosa.resample(w, orig_sr=sr, target_sr=16000)
    return np.ascontiguousarray(w)


def fbank(path: Path):
    """(T, 80) log-mel, same options as sherpa-onnx's OfflineRecognizer (samples in [-1, 1], no dither)."""
    import kaldi_native_fbank as knf
    import numpy as np

    opts = knf.FbankOptions()
    opts.frame_opts.samp_freq = 16000
    opts.frame_opts.dither = 0
    opts.frame_opts.snip_edges = False
    opts.mel_opts.num_bins = 80
    opts.mel_opts.low_freq = 20
    opts.mel_opts.high_freq = -400
    fb = knf.OnlineFbank(opts)
    fb.accept_waveform(16000, read_wave(path).tolist())
    fb.input_finished()
    return np.stack([fb.get_frame(i) for i in range(fb.num_frames_ready)]).astype(np.float32)


def _feat_job(args):
    return fbank(Path(args))


def compute_feats(cfg: dict) -> None:
    """One .npy (concatenated frames) + offsets per split part, under cfg['feat_dir']; run once, workers memory-map."""
    import numpy as np
    from multiprocessing import Pool

    split = json.loads(Path(cfg["split_json"]).read_text(encoding="utf-8"))["parts"]
    fdir = Path(cfg["feat_dir"]); fdir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with Pool(cfg.get("feat_procs", 4)) as pool:
        for part, items in split.items():
            root = Path(cfg["public_dir"] if part == "public" else cfg["train_dir"])
            feats = pool.map(_feat_job, [str(root / it["audio"]) for it in items], chunksize=16)
            np.save(fdir / f"{part}.npy", np.concatenate(feats))
            np.save(fdir / f"{part}.len.npy", np.array([len(f) for f in feats], dtype=np.int64))
            print(f"[feats] {part}: {len(items)} audio, {sum(len(f) for f in feats)} frames, {time.time() - t0:.0f}s", flush=True)


def load_feats(fdir: Path, part: str):
    import numpy as np

    flat = np.load(fdir / f"{part}.npy", mmap_mode="r")
    lens = np.load(fdir / f"{part}.len.npy")
    offs = np.concatenate([[0], np.cumsum(lens)])
    return [flat[offs[i]: offs[i + 1]] for i in range(len(lens))], lens


LOG_EPS = math.log(1e-10)  # lhotse / icefall feature padding value


def make_batches(lens, max_frames: int, rng: random.Random | None = None) -> list[list[int]]:
    """Length-bucketed batches whose padded size (longest x count) stays under max_frames; shuffled when rng is given."""
    key = [n * (rng.uniform(0.9, 1.1) if rng else 1.0) for n in lens]
    batches, cur, longest = [], [], 0
    for i in sorted(range(len(lens)), key=key.__getitem__):
        if cur and max(longest, lens[i]) * (len(cur) + 1) > max_frames:
            batches.append(cur); cur, longest = [], 0
        cur.append(i); longest = max(longest, int(lens[i]))
    if cur:
        batches.append(cur)
    if rng:
        rng.shuffle(batches)
    return batches


def collate(feats, idx, device):
    import numpy as np
    import torch

    lens = [len(feats[i]) for i in idx]
    x = np.full((len(idx), max(lens), 80), LOG_EPS, dtype=np.float32)
    for k, i in enumerate(idx):
        x[k, : lens[k]] = feats[i]
    return torch.from_numpy(x).to(device), torch.tensor(lens, dtype=torch.int64, device=device)


def spec_augment(x, lens, rng: random.Random) -> None:
    """In place, as icefall's get_spec_augment (lhotse SpecAugment, no time warp): 2 freq masks <= 27 bins, time masks
    <= 100 frames covering <= 15% of the utterance; masked with the utterance mean."""
    for b in range(x.size(0)):
        n = int(lens[b])
        mean = x[b, :n].mean()
        for _ in range(2):
            w = rng.randint(0, 27); f0 = rng.randint(0, 80 - w)
            x[b, :n, f0: f0 + w] = mean
        max_tot = 0.15 * n
        k = max(1, min(10, math.ceil(max_tot / 100)))
        for _ in range(k):
            w = rng.randint(0, int(min(100, max_tot // k))); t0 = rng.randint(0, max(0, n - w))
            x[b, t0: t0 + w] = mean


# ------------------------------------------------------------------ model (icefall egs/librispeech/ASR/zipformer)
ZIPFORMER = dict(num_encoder_layers="2,2,3,4,3,2", downsampling_factor="1,2,4,8,4,2",
                 feedforward_dim="512,768,1024,1536,1024,768", num_heads="4,4,4,8,4,4",
                 encoder_dim="192,256,384,512,384,256", query_head_dim="32", value_head_dim="12", pos_head_dim="4",
                 pos_dim=48, encoder_unmasked_dim="192,192,256,256,256,192", cnn_module_kernel="31,31,15,15,15,31",
                 decoder_dim=512, joiner_dim=512, context_size=2, causal=False, chunk_size="16,32,64,-1",
                 left_context_frames="64,128,256,-1")


def build_model(vocab_size: int, blank_id: int):
    """train.py's get_model with its default (= gipformer 1.5) sizes: transducer + CTC head, no attention decoder."""
    from decoder import Decoder
    from joiner import Joiner
    from model import AsrModel
    from scaling import ScheduledFloat
    from subsampling import Conv2dSubsampling
    from zipformer import Zipformer2

    p = ZIPFORMER
    t = lambda s: tuple(int(x) for x in str(s).split(","))
    embed = Conv2dSubsampling(in_channels=80, out_channels=t(p["encoder_dim"])[0], dropout=ScheduledFloat((0.0, 0.3), (20000.0, 0.1)))
    encoder = Zipformer2(
        output_downsampling_factor=2, downsampling_factor=t(p["downsampling_factor"]), num_encoder_layers=t(p["num_encoder_layers"]),
        encoder_dim=t(p["encoder_dim"]), encoder_unmasked_dim=t(p["encoder_unmasked_dim"]), query_head_dim=t(p["query_head_dim"]),
        pos_head_dim=t(p["pos_head_dim"]), value_head_dim=t(p["value_head_dim"]), pos_dim=p["pos_dim"], num_heads=t(p["num_heads"]),
        feedforward_dim=t(p["feedforward_dim"]), cnn_module_kernel=t(p["cnn_module_kernel"]),
        dropout=ScheduledFloat((0.0, 0.3), (20000.0, 0.1)), warmup_batches=4000.0, causal=p["causal"],
        chunk_size=t(p["chunk_size"]), left_context_frames=t(p["left_context_frames"]))
    enc_dim = max(t(p["encoder_dim"]))
    decoder = Decoder(vocab_size=vocab_size, decoder_dim=p["decoder_dim"], blank_id=blank_id, context_size=p["context_size"])
    joiner = Joiner(encoder_dim=enc_dim, decoder_dim=p["decoder_dim"], joiner_dim=p["joiner_dim"], vocab_size=vocab_size)
    return AsrModel(encoder_embed=embed, encoder=encoder, decoder=decoder, joiner=joiner, attention_decoder=None,
                    encoder_dim=enc_dim, decoder_dim=p["decoder_dim"], vocab_size=vocab_size, use_transducer=True,
                    use_ctc=True, use_attention_decoder=False)


def set_batch_count(model, count: float) -> None:
    for m in model.modules():
        if hasattr(m, "batch_count"):
            m.batch_count = count


def write_cache(path: Path, tag: str, items, hyp):
    """Same layout as lmlib.asr legacy caches (header row, then one record per audio), readable by load_cache."""
    with open(path, "w", encoding="utf-8") as f:
        f.write(json.dumps({"_header": {"run_hash": tag, "created": time.time()}}) + "\n")
        for it in items:
            f.write(json.dumps({"id": it["audio"], "status": "completed", "text": hyp[it["audio"]]}, ensure_ascii=False) + "\n")


# ------------------------------------------------------------------ worker
def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--variant")
    ap.add_argument("--feats", action="store_true", help="precompute features for every split part, then exit")
    ap.add_argument("--smoke", action="store_true", help="load + 2 train steps + decode a few dev audio, then exit")
    a = ap.parse_args(argv)
    cfg = json.loads(Path(a.config).read_text(encoding="utf-8"))
    if a.feats:
        return compute_feats(cfg)
    v = cfg["variants"][a.variant]
    sys.path[:0] = [cfg["icefall_dir"], str(Path(cfg["icefall_dir"]) / "egs/librispeech/ASR/zipformer")]

    import k2
    import numpy as np
    import sentencepiece as spm
    import torch
    from beam_search import greedy_search_batch, modified_beam_search
    from icefall.utils import create_grad_scaler, get_parameter_groups_with_lrs
    from optim import Eden, ScaledAdam

    device = torch.device("cuda")
    random.seed(cfg["seed"]); np.random.seed(cfg["seed"]); torch.manual_seed(cfg["seed"])
    out = Path(cfg["out_dir"]) / a.variant
    out.mkdir(parents=True, exist_ok=True)
    split = json.loads(Path(cfg["split_json"]).read_text(encoding="utf-8"))["parts"]
    fdir = Path(cfg["feat_dir"])
    dev = split["dev_seen"] + split["dev_unseen"]
    deadline = cfg["session_started"] + cfg["session_budget_s"]
    metrics = {"variant": a.variant, "config": v, "epochs": []}

    def save_metrics():
        (out / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=1), encoding="utf-8")

    t = time.time()
    mdir = Path(cfg["model_dir"])
    sp = spm.SentencePieceProcessor(); sp.load(str(mdir / "bpe.model"))
    blank_id, unk_id = sp.piece_to_id("<blk>"), sp.piece_to_id("<unk>")
    model = build_model(sp.get_piece_size(), blank_id)
    state = torch.load(mdir / "model.pt", map_location="cpu", weights_only=False)
    model.load_state_dict(state.get("model", state), strict=True)
    model.to(device)
    n_params = sum(p.numel() for p in model.parameters())
    if n_params + 4022468096 >= 6_000_000_000:
        raise ValueError("LifeMate total parameter limit exceeded")

    fe = {p: load_feats(fdir, p) for p in ("train_fit", "dev_seen", "dev_unseen", "final_holdout", "public")}
    dev_feats = fe["dev_seen"][0] + fe["dev_unseen"][0]
    dev_lens = np.concatenate([fe["dev_seen"][1], fe["dev_unseen"][1]])

    # Train targets: spoken form, upper case; drop audio with <unk> pieces or more tokens than encoder frames (CTC inf)
    tr_items, tr_feats, tr_lens, tr_tok, dropped = [], [], [], [], Counter()
    for it, f, n in zip(split["train_fit"], *fe["train_fit"]):
        ids = sp.encode(target_text(it["text"]), out_type=int)
        if unk_id in ids:
            dropped["unk"] += 1; continue
        if not ids or ((int(n) - 7) // 2 + 1) // 2 < len(ids):
            dropped["too_short"] += 1; continue
        tr_items.append(it); tr_feats.append(f); tr_lens.append(int(n)); tr_tok.append(ids)
    metrics.update(model_load_s=time.time() - t, parameter_count=n_params, model=cfg["model"], revision=cfg["revision"],
                   n_train=len(tr_items), train_dropped=dict(dropped), n_dev=len(dev))
    print(f"[gipft {a.variant}] loaded {n_params / 1e6:.1f}M params in {metrics['model_load_s']:.0f}s; train {len(tr_items)} audio "
          f"(dropped {dict(dropped)}), dev {len(dev)}", flush=True)

    def decode(feats, lens, method="greedy") -> list[str]:
        model.eval(); res = {}
        order = make_batches(lens, v["eval_max_frames"])
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for idx in order:
                x, xl = collate(feats, idx, device)
                enc, enc_lens = model.forward_encoder(x, xl)
                if method == "greedy":
                    hyps = greedy_search_batch(model=model, encoder_out=enc, encoder_out_lens=enc_lens)
                else:
                    hyps = modified_beam_search(model=model, encoder_out=enc, encoder_out_lens=enc_lens, beam=4)
                for i, h in zip(idx, hyps):
                    res[i] = sp.decode(h)
        return [res[i] for i in range(len(feats))]

    def eval_dev(epoch, train_s=None, loss=None):
        te = time.time()
        texts = decode(dev_feats, dev_lens)
        hyp = {it["audio"]: s for it, s in zip(dev, texts)}
        rec = {"epoch": epoch, "dev": score(dev, hyp), "s_per_audio": (time.time() - te) / len(dev), "train_s": train_s,
               "train_loss": loss, "peak_vram_mb": torch.cuda.max_memory_allocated() / 2**20}
        metrics["epochs"].append(rec); save_metrics()
        d = rec["dev"]
        print(f"[gipft {a.variant}] epoch {epoch}: dev WER all {d['all']['wer']:.4f} final {d['final']['wer']:.4f} "
              f"(seen {d['dev_seen/final']['wer']:.4f} unseen {d['dev_unseen/final']['wer']:.4f}; raw final {d['final']['wer_raw']:.4f}) "
              f"{rec['s_per_audio']:.3f}s/audio", flush=True)
        return d["all"]["wer"], hyp

    rng = random.Random(cfg["seed"])
    opt = ScaledAdam(get_parameter_groups_with_lrs(model, lr=v["lr"], include_names=True), lr=v["lr"], clipping_scale=2.0)
    sched = Eden(opt, v.get("lr_batches", 100000.0), v.get("lr_epochs", 100.0), warmup_batches=v.get("warmup_batches", 100.0))
    scaler = create_grad_scaler(enabled=True, init_scale=1.0)
    step = 0

    def train_step(idx) -> tuple[float, int]:
        nonlocal step
        if step % 10 == 0:  # ScheduledFloat (dropout, skip rates) at their end values, as icefall finetune.py
            set_batch_count(model, 100000 + step * v["max_frames"] / 60000)
        step += 1
        x, xl = collate(tr_feats, idx, device)
        spec_augment(x, xl, rng)
        y = k2.RaggedTensor([tr_tok[i] for i in idx])
        with torch.autocast("cuda", dtype=torch.float16):
            simple, pruned, ctc = model(x=x, x_lens=xl, y=y, prune_range=5, am_scale=0.0, lm_scale=0.25)[:3]
            loss = v.get("simple_loss_scale", 0.5) * simple + pruned + v.get("ctc_loss_scale", 0.2) * ctc
        scaler.scale(loss).backward()
        sched.step_batch(step)
        scaler.step(opt); scaler.update(); opt.zero_grad()
        if step % 100 == 0:  # as icefall: the sum-reduced loss starts at scale 1.0, grow it while it is small
            cur = scaler._scale.item()
            if cur < 8.0 or (cur < 32.0 and step % 400 == 0):
                scaler.update(cur * 2.0)
        return loss.item(), int(((xl - 7) // 2 + 1).div(2, rounding_mode="floor").sum())

    if a.smoke:
        model.train()
        for idx in make_batches(tr_lens, v["max_frames"], rng)[:2]:
            print("[smoke] train step loss/frame", (lambda r: r[0] / r[1])(train_step(idx)), flush=True)
        few = list(range(16))
        texts = decode([dev_feats[i] for i in few], dev_lens[few])
        for i, s in zip(few[:5], texts):
            print("[smoke]", dev[i]["text"], "->", s, flush=True)
        print("[smoke] ok, peak MB", torch.cuda.max_memory_allocated() / 2**20, flush=True)
        return

    best_wer, best_hyp = eval_dev(0) if cfg.get("eval_zero_shot", True) else (float("inf"), None)
    best_epoch, best_state = 0, None
    epoch_s = None
    for ep in range(1, v["epochs"] + 1):
        if epoch_s and time.time() + 1.3 * epoch_s + cfg["reserve_s"] > deadline:
            print(f"[gipft {a.variant}] stop before epoch {ep}: session budget", flush=True)
            metrics["stopped_by_budget"] = ep; break
        model.train(); te = time.time(); tot = frames = 0.0
        sched.step_epoch(ep - 1)
        batches = make_batches(tr_lens, v["max_frames"], rng)
        for b, idx in enumerate(batches):
            loss, nf = train_step(idx)
            if not math.isfinite(loss):
                print(f"[gipft {a.variant}] ep {ep} step {b}: loss {loss} (GradScaler skips the step)", flush=True)
                continue
            tot += loss; frames += nf
            if b % 20 == 0:
                print(f"[gipft {a.variant}] ep {ep} step {b}/{len(batches)} loss/frame {tot / max(frames, 1):.4f} "
                      f"lr {sched.get_last_lr()[0]:.2e}", flush=True)
        wer, hyp = eval_dev(ep, time.time() - te, tot / max(frames, 1))
        epoch_s = time.time() - te
        if cfg.get("full_data") or wer < best_wer:  # full_data: dev nằm trong train nên luôn lấy epoch cuối
            best_wer, best_epoch, best_hyp = wer, ep, hyp
            best_state = {k: t.detach().to("cpu", copy=True) for k, t in model.state_dict().items()}

    metrics.update(best_epoch=best_epoch, best_dev_wer=best_wer)
    if best_state is not None:
        model.load_state_dict(best_state)
    elif metrics["epochs"][-1]["epoch"] > 0:  # zero-shot stayed best: back to the released weights
        model.load_state_dict(state.get("model", state))
    try:  # beam search on dev for information only (cached transcripts stay greedy, as the agent pipeline decodes)
        texts = decode(dev_feats, dev_lens, "beam")
        metrics["dev_beam4"] = score(dev, {it["audio"]: s for it, s in zip(dev, texts)})
    except Exception as e:  # noqa: BLE001
        metrics["dev_beam4_error"] = f"{type(e).__name__}: {e}"
    hold_items = split["final_holdout"]
    hold = dict(zip([it["audio"] for it in hold_items], decode(*fe["final_holdout"])))
    metrics["holdout"] = score(hold_items, hold)
    tp = time.time()
    pub = dict(zip([it["audio"] for it in split["public"]], decode(*fe["public"])))
    metrics["public_s_per_audio"] = (time.time() - tp) / max(len(pub), 1)
    tag = f"gipft-{'full-' if cfg.get('full_data') else ''}{a.variant}-ep{best_epoch}"
    write_cache(out / f"asr_train_gipft_{a.variant}.shard0.jsonl", tag, dev + hold_items, {**best_hyp, **hold})
    write_cache(out / f"asr_public_test_gipft_{a.variant}.shard0.jsonl", tag, split["public"], pub)
    if cfg.get("full_data"):
        beam = dict(zip([it["audio"] for it in split["public"]], decode(*fe["public"], "beam")))
        write_cache(out / f"asr_public_test_gipft_{a.variant}_beam.shard0.jsonl", tag + "-beam4", split["public"], beam)
    if best_state is not None and cfg.get("save_model", True):
        torch.save({"model": best_state}, out / "model_best.pt")  # icefall format: export-onnx.py can read it
    metrics["total_s"] = time.time() - cfg["session_started"]
    save_metrics()
    h = metrics["holdout"]
    print(f"[gipft {a.variant}] best epoch {best_epoch}: dev WER {best_wer:.4f}; holdout WER all {h['all']['wer']:.4f} "
          f"final {h['final']['wer']:.4f}; public {len(pub)} audio at {metrics['public_s_per_audio']:.3f}s/audio", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
sys.path.insert(0, str(WORK))
import gipft
load = lambda p: [json.loads(l) for l in open(p, encoding="utf-8")]
split = gipft.build_items(load(TRAIN_DIR / "train.jsonl"), load(PUBLIC_DIR / "public_test.jsonl"))
# Cùng split seed 2026 với mọi run agent (lmlib.data.make_manifest) và lm-A-030
assert split["manifest_counts"] == {"train_fit": 974, "dev_seen": 154, "final_holdout": 154, "dev_unseen": 263}, split["manifest_counts"]
# Toàn bộ audio train; các phần dev vẫn giữ để theo dõi loss không vỡ, không phải đo độc lập
split["parts"]["train_fit"] = split["parts"]["train_fit"] + split["parts"]["dev_seen"] + split["parts"]["dev_unseen"] + split["parts"]["final_holdout"]
assert len(split["parts"]["train_fit"]) == 5036 and len(split["parts"]["public"]) == 1933, {k: len(v) for k, v in split["parts"].items()}
print(split["holdout_apps"], {k: len(v) for k, v in split["parts"].items()})
for t in ["Nhắc chị ra sân bay lúc 8 giờ kém 15 tối mai nha.", "Xăng RON95 giờ bao nhiêu?", "gọi số 0983 214 6755"]:
    print(t, "→", gipft.target_text(t))
(OUT / "split.json").write_text(json.dumps(split, ensure_ascii=False), encoding="utf-8")
cfg = dict(CONFIG, split_json=str(OUT / "split.json"), train_dir=str(TRAIN_DIR), public_dir=str(PUBLIC_DIR), model_dir=MODEL_DIR,
           session_started=SESSION_STARTED)
(OUT / "config.json").write_text(json.dumps(cfg, indent=1), encoding="utf-8")
# fbank một lần cho mọi phần (train / dev / holdout / public), worker đọc bằng memory-map
assert run_py([str(WORK / "gipft.py"), "--config", str(OUT / "config.json"), "--feats"]) == 0

In [ ]:
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1")
assert run_py([str(WORK / "gipft.py"), "--config", str(OUT / "config.json"), "--variant", "lr15", "--smoke"], env=env) == 0, "smoke lỗi, xem log"

In [ ]:
name, v = "lr15", CONFIG["variants"]["lr15"]
(OUT / name).mkdir(exist_ok=True)
log = open(OUT / name / "worker.log", "a", encoding="utf-8")
env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(v["gpu"]), PYTHONUNBUFFERED="1", OMP_NUM_THREADS="1")
p = subprocess.Popen([sys.executable, str(WORK / "gipft.py"), "--config", str(OUT / "config.json"), "--variant", name], env=env, stdout=log, stderr=subprocess.STDOUT, cwd=str(WORK))
while p.poll() is None:
    time.sleep(120)
    lines = [l for l in open(OUT / name / "worker.log", encoding="utf-8", errors="replace") if l.startswith("[gipft")]
    print(f"{time.time() - SESSION_STARTED:7.0f}s {name}: {lines[-1].strip() if lines else '(đang load)'}", flush=True)
print("".join(open(OUT / name / "worker.log", encoding="utf-8", errors="replace").readlines()[-12:]))
if p.returncode:
    raise RuntimeError(f"Worker lỗi {p.returncode}")


In [ ]:
# Không có dev độc lập: chỉ kiểm train xong đủ epoch và đủ transcript public
m = json.loads((OUT / "lr15" / "metrics.json").read_text())
print({k: m.get(k) for k in ("n_train", "train_dropped", "best_epoch", "parameter_count", "public_s_per_audio", "total_s")})
print("loss theo epoch", [(e["epoch"], round(e["train_loss"], 4)) for e in m["epochs"] if e["train_loss"] is not None])
if m.get("best_epoch") != CONFIG["variants"]["lr15"]["epochs"] or m.get("stopped_by_budget"):
    raise RuntimeError("Chưa train đủ epoch")
for f in ("asr_public_test_gipft_lr15.shard0.jsonl", "asr_public_test_gipft_lr15_beam.shard0.jsonl"):
    lines = (OUT / "lr15" / f).read_text(encoding="utf-8").splitlines()
    if len(lines) != 1934:
        raise RuntimeError(f"{f}: {len(lines)} dòng")
    print(f, json.loads(lines[0]), json.loads(lines[1])["text"][:90])
shutil = __import__("shutil"); shutil.rmtree(CONFIG["feat_dir"], ignore_errors=True)
